# ColdSite-DTI on KIBA — closing the asymmetry

KIBA was trained for the two **published** models and the accuracy anchor; our own model
was cut to fit the compute. The paper therefore says, in Limitations, that our model is
audited on one dataset where the models whose claims the paper is about are audited on
two. A reviewer is entitled to read that as convenient. This notebook removes it: the
**six cells** ColdSite-DTI is missing — random and cold-drug, three seeds — on the
identical splits every other model faced.

| | |
|---|---|
| model | ColdSite-DTI (ours), trained exactly as its DAVIS cells were |
| cells | 2 levels × 3 seeds = 6 |
| precision | **full (fp32)**, see below |
| cost | 3.6 h per cell at the 25-epoch floor, 5.2 h at the 36-epoch median (measured on a T4, `results/speed_test_kiba_t4.md`) |
| plan | 3 cells per GPU, so **11–16 h per GPU: expect two commits** |

**Why full precision.** Mixed precision would buy 15% on this model (0.387 → 0.336
s/batch) and cost the thing that matters more: its DAVIS cells are fp32, and a model
trained in one precision on one dataset and another precision on the other is not the
clean replication the paper claims. MolTrans's KIBA cells were kept fp32 for the same
reason — there because float16 made it diverge, here because consistency is worth more
than 15%.

**Two commits are expected, not a failure.** The runner self-stops at 11 hours so the
commit can save its output. Whatever is unfinished continues from its last finished epoch
on the next commit: download the output, make it a private dataset, attach it, set
`RESTORE_FROM = '/kaggle/input'` in section 1, and run again.

## What to do

1. **Settings** (right panel): Accelerator **GPU T4 x2**, Internet **On**.
2. Run all. Nothing to edit on the first commit.
3. **Save Version → Save & Run All (Commit)**.


## 1. Settings — only RESTORE_FROM, and only on a later commit

In [ ]:
# ============================================================================
# SETTINGS
# ============================================================================

RESTORE_FROM = None      # second commit onward: '/kaggle/input' (searches every input)

# ============================================================================
# Everything below is the plan. Read it; do not edit it.
# ============================================================================

DATASET = 'kiba'
TASK = 'binary'
BRANCH = 'main'
MODEL = 'coldsite_dti'
LEVELS = ['random', 'cold_drug']
SEEDS = [1, 2, 3]
EPOCHS = 100
AMP = False              # see the header: its DAVIS cells are fp32 and must stay comparable

# Measured peak memory on a 1,000-residue protein: 8.7 GB at batch 64, 2.3 GB at 16
# (STATUS.md, from the DAVIS grid). A T4 has 15.6 GB, so 64 fits with room.
BATCH_SIZE = 64

CELLS = [(level, seed) for level in LEVELS for seed in SEEDS]
QUEUE0 = [c for i, c in enumerate(CELLS) if i % 2 == 0]
QUEUE1 = [c for i, c in enumerate(CELLS) if i % 2 == 1]
TOTAL_CELLS = len(CELLS)

print(f'{TOTAL_CELLS} cells: ColdSite-DTI x {len(LEVELS)} levels x {len(SEEDS)} seeds')
print('GPU 0:', QUEUE0)
print('GPU 1:', QUEUE1)
print(f'precision: {"mixed" if AMP else "full (fp32)"} | batch {BATCH_SIZE}')


## 2. Check the GPU(s)

In [ ]:
import time
START = time.time()          # the 11-hour self-stop is measured from here

import torch

assert torch.cuda.is_available(), 'No CUDA. Settings -> Accelerator -> GPU.'
N_GPU = torch.cuda.device_count()
for i in range(N_GPU):
    p = torch.cuda.get_device_properties(i)
    print(f'GPU {i}: {p.name}, {p.total_memory/1e9:.1f} GB')
print('torch  :', torch.__version__)

if torch.cuda.get_device_properties(0).total_memory / 1e9 < 14:
    BATCH_SIZE = 16
    print(f'small GPU -- batch size lowered to {BATCH_SIZE} (8.7 GB at 64 would not fit)')


## 3. Clone the repo

No vendored model to install: ColdSite-DTI is ours and lives in `src/`.

In [ ]:
import os

REPO = 'https://github.com/Mahim56207/ColdSite-DTI_New.git'
WORK = '/kaggle/working'
SRC  = f'{WORK}/ColdSite-DTI_New'

if not os.path.exists(SRC):
    !git clone --branch {BRANCH} {REPO} {SRC}
os.chdir(SRC)
!git checkout {BRANCH}
!git pull origin {BRANCH}
!pip install -q tabulate

for _needed in ('src/model/run_grid.py', 'src/model/train.py',
                'src/model/resume.py', 'src/model/precision.py'):
    assert os.path.exists(_needed), (
        f'{_needed} is missing from branch {BRANCH!r}. Without resume.py a cell longer '
        'than one 11-hour commit could never finish.')

RESULTS = f'{WORK}/results'
os.makedirs(RESULTS, exist_ok=True)
print()
!git log --oneline -1
print('results ->', RESULTS)


## 4. Fetch the KIBA source files

`build_splits` reads DeepDTA's published files, which `.gitignore` keeps out of the repo.
Both datasets are fetched because `load_data` reads both.

In [ ]:
BASE = 'https://raw.githubusercontent.com/hkmztrk/DeepDTA/master/data'
for ds in ('davis', 'kiba'):
    os.makedirs(f'src/data/baselines/deepdta/data/{ds}', exist_ok=True)
    for fname in ('ligands_can.txt', 'proteins.txt', 'Y'):
        target = f'src/data/baselines/deepdta/data/{ds}/{fname}'
        if not os.path.exists(target):
            !curl -sL {BASE}/{ds}/{fname} -o {target}

missing = [f'{ds}/{f}' for ds in ('davis', 'kiba')
           for f in ('ligands_can.txt', 'proteins.txt', 'Y')
           if not os.path.exists(f'src/data/baselines/deepdta/data/{ds}/{f}')
           or os.path.getsize(f'src/data/baselines/deepdta/data/{ds}/{f}') == 0]
assert not missing, (f'these source files did not download: {missing}. '
                     'Internet must be ON (Settings -> Internet).')
print('DeepDTA source files present')

!python -m src.data.load_data


## 5. Build the splits — and verify they match

The point of this notebook is that our model faces the identical test the published ones
faced. That is only true if these numbers match, so a mismatch stops the run.

In [ ]:
import subprocess, sys

import pandas as pd
from src.model.dataset import BINARY_THRESHOLD

built = subprocess.run([sys.executable, '-m', 'src.data.build_splits'],
                       capture_output=True, text=True)
for line in built.stdout.splitlines():
    if 'kiba' in line or 'leakage' in line:
        print(line)
assert built.returncode == 0, (
    'build_splits failed:\n' + (built.stderr or built.stdout)[-1500:] +
    '\n\nIf it names a missing source file, section 4 did not download it.')

# The same figures the four-account KIBA notebook checked, from data/splits/kiba.
EXPECTED = {
    'random':      (82778, 11825, 23651),
    'cold_drug':   (83807, 12073, 22374),
    'cold_target': (85452, 10701, 22101),
    'cold_pair':   (58041,  1334,  4375),
}
assert BINARY_THRESHOLD['kiba'] == 12.1, BINARY_THRESHOLD
print(f"\nbinary threshold: KIBA score >= {BINARY_THRESHOLD['kiba']}")

ok = True
for level, expected in EXPECTED.items():
    sizes = tuple(len(pd.read_csv(f'data/splits/{DATASET}/{level}/{part}.csv'))
                  for part in ('train', 'valid', 'test'))
    mark = 'OK' if sizes == expected else f'MISMATCH, expected {expected}'
    ok &= sizes == expected
    print(f'{level:12s} {str(sizes):28s} {mark}')
assert ok, ('splits differ from the ones the other models trained on -- stop. Whatever '
            'this notebook produced would not be comparable with Results section 8.')
print('\nsplits match the record: this model faces the same test.')


## 6. Restore from a previous commit

Two commits are expected. On the second, attach this account's own output as a private
dataset and set `RESTORE_FROM = '/kaggle/input'` in section 1. Finished cells are skipped
and an interrupted one continues from its last finished epoch.

In [ ]:
import shutil, glob

if RESTORE_FROM:
    assert os.path.isdir(RESTORE_FROM), f'not a directory: {RESTORE_FROM}'
    copied = skipped = 0
    for src in glob.glob(f'{RESTORE_FROM}/**/*', recursive=True):
        name = os.path.basename(src)
        if not name.endswith(('.pt', '_results.json', '_history.json')):
            continue
        dst = os.path.join(RESULTS, name)
        if os.path.exists(dst):
            skipped += 1
            continue
        shutil.copy2(src, dst)
        copied += 1
    print(f'restored {copied} file(s), left {skipped} already present')
else:
    print('RESTORE_FROM is None -- starting from an empty results folder.')


## 7. The runner

Two queues, one per GPU, each cell its own process. A STATUS line every 10 minutes carries
the measured minutes-per-epoch against the projection and says whether the queue fits
inside this commit. Self-stops at 11 hours.

In [ ]:
import glob, json, os, re, subprocess, threading, time
from src.model.checkpoint_naming import checkpoint_path, results_path, run_tag

DEADLINE = START + 11 * 3600
STATUS_EVERY = 600          # seconds between STATUS lines

QUEUE_OF = {'0': QUEUE0, '1': QUEUE1}
if N_GPU < 2:                      # one GPU: the same work, one queue, twice the wall time
    QUEUE_OF = {'0': QUEUE0 + QUEUE1}
    print('WARNING: one GPU. Six cells in a single queue will need three commits, not two.')

MODEL_NAME = {'src.model.run_grid': 'ColdSite'}
MODEL_KEY = {'src.model.run_grid': 'coldsite_dti'}

# Measured on a T4, full precision (results/speed_test_kiba_t4.md): 7.5 min/epoch, so
# 3.6 h at the 25-epoch floor and 5.2 h at the 36-epoch DAVIS median. The STATUS line
# prints the measured rate beside this projection with their ratio.
MIN_PER_EPOCH = {'coldsite_dti': 8.7}
EPOCHS_MIN, EPOCHS_TYPICAL = 25, 36
HEADER = re.compile(r'^\w+/(\w+)/seed(\d+)\s*$')          # run_grid: a cell starts
SKIPPED = re.compile(r'^\[skip\] \w+/(\w+)/seed(\d+)')     # run_grid: a cell was done
EPOCH = re.compile(r'^\s*(?:epoch|Epoch)\s+(\d+)')
SAVED = re.compile(r'Saved -> (\S+_results\.json)')


def hours_left():
    return (DEADLINE - time.time()) / 3600


def note_epoch(w, number, now):
    """Record an epoch boundary and keep a mean seconds-per-epoch.

    Called for every line that looks like an epoch header. The baselines print
    "epoch 1 train batch 7/10 ..." for every batch, so the same number arrives many
    times and only a CHANGE is a boundary. The first epoch of a cell also carries the
    split encoding (KIBA is 83,000 rows), so it is the starting mark but its own
    duration is never counted in the rate.
    """
    if number != w['last_epoch']:
        gap = number - w['last_epoch'] if w['last_epoch'] else 0
        if gap > 0 and w['last_epoch_at'] is not None:
            # Divide by the gap: a resumed cell's first reported epoch is not 1, and one
            # elapsed stretch may cover several epochs. Weighted mean, so a stretch of
            # three epochs counts three times as much as a single one.
            per = (now - w['last_epoch_at']) / gap
            n = w['timed_epochs']
            w['sec_per_epoch'] = (per if not n
                                  else (w['sec_per_epoch'] * n + per * gap) / (n + gap))
            w['timed_epochs'] = n + gap
        w['last_epoch'], w['last_epoch_at'] = number, now
    w['epoch'] = str(number)
    return w


def eta(w, now, deadline):
    """Two lines about one GPU: the measured rate, and when it finishes.

    `w` is that GPU's live state. Returns [] until a rate exists -- claiming an ETA from
    a single epoch would mean quoting the split-encoding time as the epoch time.
    """
    rate = w.get('sec_per_epoch')
    key, index = w.get('key'), w.get('cell', 0)
    if not rate or not key:
        return []
    predicted = MIN_PER_EPOCH[key] * 60
    drift = rate / predicted if predicted else float('nan')
    done = int(w.get('epoch') or 0)
    lines = [f"      {rate / 60:.1f} min/epoch measured, {predicted / 60:.1f} projected "
             f"(x{drift:.2f}) over {w.get('timed_epochs', 0)} epoch(s)"]

    # this cell, if it stops at the floor or at the DAVIS median
    at = []
    for label, total in (('min', EPOCHS_MIN), ('median', EPOCHS_TYPICAL)):
        left = max(total - done, 0) * rate
        at.append(f"{label} {time.strftime('%H:%M', time.localtime(now + left))}"
                  f" (+{left / 3600:.1f} h)")
    lines.append(f"      this cell ends: {' | '.join(at)}")

    # the rest of this GPU's queue, rescaled by the drift we are actually seeing
    queued = ORDERED.get(str(w.get('gpu')), [])
    remaining = queued[index:]                      # cells not started yet
    if remaining:
        rest = sum(HOURS[m][0] for m, _lv, _s in remaining) * drift
        this_cell = max(EPOCHS_TYPICAL - done, 0) * rate / 3600
        total_left = rest + this_cell
        verdict = ('inside this commit' if now + total_left * 3600 <= deadline
                   else f'needs about {int(total_left / 11) + 1} more commit(s)')
        lines.append(f"      queue: {len(remaining)} cell(s) after this one, "
                     f"~{total_left:.1f} h left at the median -> {verdict}")
    else:
        this_cell = max(EPOCHS_TYPICAL - done, 0) * rate / 3600
        verdict = ('inside this commit' if now + this_cell * 3600 <= deadline
                   else 'needs another commit')
        lines.append(f"      last cell of this queue, ~{this_cell:.1f} h left at the "
                     f"median -> {verdict}")
    return lines


def cells_done():
    """This account's finished cells. Counting a MODELS x LEVELS x SEEDS product would
    count cells another account owns and report progress that is not ours."""
    done = 0
    for queue in MY_CELLS.values():
        for model, split, seed in queue:
            tag = run_tag(DATASET, split, TASK, seed)
            if os.path.exists(results_path(RESULTS, tag, model=model)):
                done += 1
    return done


def same_as_other_seed(results_file):
    """Another seed of this model and split with exactly the same test metrics means the
    seed never reached training -- MolTrans's vendored import reseeded torch with 1, so
    its three DAVIS seeds were one run three times (2026-09-13)."""
    try:
        mine = json.load(open(results_file))['test_metrics']
        for other in glob.glob(re.sub(r'_seed\d+', '_seed[0-9]', results_file)):
            if other != results_file and json.load(open(other))['test_metrics'] == mine:
                return (f'SEEDS IDENTICAL: same test metrics as {os.path.basename(other)} '
                        f'-- is --seed reaching training? Stop and check before going on.')
    except Exception:
        return None
    return None

def _arg(cmd, flag):
    return cmd[cmd.index(flag) + 1] if flag in cmd else '-'


def _cells_in(cmd):
    if cmd[3] == 'src.model.run_grid':
        return len(_arg(cmd, '--splits').split(',')) * len(_arg(cmd, '--seeds').split(','))
    return 1


def run_parallel(queues, label):
    """queues: {gpu: [command, ...]}. Returns True if the deadline cut it short."""
    current, state = {}, {'deadline': False}
    where = {g: {'model': '-', 'split': '-', 'seed': '-', 'epoch': '-', 'cell': 0,
                 'total': sum(_cells_in(c) for c in q), 'gpu': g, 'key': None,
                 'sec_per_epoch': None, 'timed_epochs': 0, 'last_epoch': None,
                 'last_epoch_at': None}
             for g, q in queues.items()}

    def tag(g):
        w = where[g]
        return f"[GPU{g} {w['model']} {w['split']} s{w['seed']} · cell {w['cell']}/{w['total']}]"

    def begin_cell(g, split, seed):
        where[g].update(split=split, seed=seed, epoch='-', sec_per_epoch=None,
                        timed_epochs=0, last_epoch=None, last_epoch_at=None)
        where[g]['cell'] += 1

    def worker(gpu, commands):
        env = {**os.environ, 'CUDA_VISIBLE_DEVICES': gpu, 'PYTHONUNBUFFERED': '1'}
        with open(f'{WORK}/{label}_gpu{gpu}.log', 'a') as log:
            for cmd in commands:
                if time.time() > DEADLINE:
                    return
                module = cmd[3]
                where[gpu]['model'] = MODEL_NAME.get(module, module)
                where[gpu]['key'] = MODEL_KEY.get(module)
                if module != 'src.model.run_grid':          # one command = one cell
                    begin_cell(gpu, _arg(cmd, '--split'), _arg(cmd, '--seed'))
                proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE,
                                        stderr=subprocess.STDOUT, text=True, bufsize=1)
                current[gpu] = proc
                for line in proc.stdout:
                    if module == 'src.model.run_grid':      # follow run_grid's own cells
                        m = HEADER.match(line) or SKIPPED.match(line)
                        if m:
                            begin_cell(gpu, m.group(1), m.group(2))
                    m = EPOCH.match(line)
                    if m:
                        note_epoch(where[gpu], int(m.group(1)), time.time())
                    print(f'{tag(gpu)} {line}', end='', flush=True)
                    log.write(line)
                    log.flush()
                    m = SAVED.search(line)
                    if m:
                        try:
                            auc = json.load(open(m.group(1)))['test_metrics'].get('auroc')
                            auc = f'{auc:.4f}'
                        except Exception:
                            auc = '?'
                        print(f'  ✓ {tag(gpu)} finished -- test AUROC {auc}   '
                              f'[{cells_done()}/{TOTAL_CELLS} cells complete]', flush=True)
                        warning = same_as_other_seed(m.group(1))
                        if warning:
                            print(f'  !! {tag(gpu)} {warning}', flush=True)
                code_ = proc.wait()
                if code_ != 0 and not state['deadline']:
                    print(f'{tag(gpu)} !! exited {code_}: {" ".join(cmd[3:])}', flush=True)

    threads = [threading.Thread(target=worker, args=(g, q), daemon=True)
               for g, q in queues.items()]
    for t in threads:
        t.start()
    last_status = 0.0
    while any(t.is_alive() for t in threads):
        if time.time() - last_status >= STATUS_EVERY:
            last_status = time.time()
            now_ = time.time()
            print(f"\n=== STATUS {time.strftime('%H:%M')} | "
                  f"{cells_done()}/{TOTAL_CELLS} cells complete | "
                  f"{(now_ - START) / 3600:.1f} h in, {hours_left():.1f} h before the "
                  f"stop ===", flush=True)
            for g, w in sorted(where.items()):
                print(f"   GPU{g}: {w['model']} {w['split']} s{w['seed']} "
                      f"cell {w['cell']}/{w['total']} epoch {w['epoch']}", flush=True)
                for line in eta(w, now_, DEADLINE):
                    print(line, flush=True)
            print(flush=True)
        if time.time() > DEADLINE and not state['deadline']:
            state['deadline'] = True
            print('\n*** 11-hour mark: stopping so this commit can save its output. '
                  'Unfinished cells continue from their last finished epoch next commit. ***\n', flush=True)
            for proc in list(current.values()):
                if proc.poll() is None:
                    proc.terminate()
        time.sleep(15)
    return state['deadline']


def coldsite_cmd(level, seed):
    """One cell. ColdSite-DTI has no single-cell trainer, so it goes through run_grid with
    one split and one seed -- which is also how its DAVIS cells were trained, so the
    recipe is unchanged. Its patience is fixed at 15 inside run_training and is not a
    flag; --skip-if-done is likewise run_grid's own behaviour, which is why finished cells
    are skipped on a second commit without passing anything."""
    return ['python', '-u', '-m', 'src.model.run_grid',
            '--datasets', DATASET, '--splits', level, '--seeds', str(seed),
            '--task', TASK, '--epochs', str(EPOCHS), '--min-epochs', '10',
            '--batch-size', str(BATCH_SIZE), '--results-dir', RESULTS] + (['--amp'] if AMP else [])


# Both KIBA levels are within 1,000 rows of each other, so the queues are balanced by
# construction; seeds are spread across GPUs so a GPU dying does not cost a whole level.
TRAIN_ROWS = {'random': 82778, 'cold_drug': 83807}
HOURS = {'coldsite_dti': (EPOCHS_TYPICAL * MIN_PER_EPOCH['coldsite_dti'] / 60,
                          EPOCHS_MIN * MIN_PER_EPOCH['coldsite_dti'] / 60)}

ORDERED = {gpu: [(MODEL, level, seed)
                 for level, seed in sorted(cells, key=lambda c: -TRAIN_ROWS[c[0]])]
           for gpu, cells in sorted(QUEUE_OF.items())}
QUEUES = {gpu: [coldsite_cmd(level, seed) for _model, level, seed in cells]
          for gpu, cells in sorted(ORDERED.items())}
MY_CELLS = ORDERED

for gpu, cells in sorted(ORDERED.items()):
    print(f'GPU {gpu}: in this order')
    for _model, level, seed in cells:
        print(f'   coldsite_dti  {level:10s} seed {seed}   {TRAIN_ROWS[level]:,} train rows  '
              f'~{HOURS["coldsite_dti"][1]:.1f}-{HOURS["coldsite_dti"][0]:.1f} h')
print(f'{hours_left():.1f} h left before the self-stop')


## 8. Launch

In [ ]:
if hours_left() < 0.5:
    raise SystemExit('less than 30 minutes before the self-stop -- not worth starting')

cut = run_parallel(QUEUES, f'coldsite_{DATASET}')
print()
print(f'{cells_done()}/{TOTAL_CELLS} cells complete')
if cut:
    print('CUT SHORT by the 11-hour stop -- expected on the first commit. Download the '
          "output, make it a private dataset, attach it, set RESTORE_FROM = "
          "'/kaggle/input' in section 1, and run again.")
else:
    print('every cell finished')


## 9. What landed

Believable for ColdSite-DTI on KIBA: AUROC around 0.85–0.93 at random and lower on unseen
drugs. Its DAVIS binary cells were 0.884–0.906 at random. 0.5 means it never learned;
above 0.98 means look for leakage before celebrating.

In [ ]:
import glob, json

rows = []
for path in sorted(glob.glob(f'{RESULTS}/kiba_*_binary_seed*_results.json')):
    if any(tag in os.path.basename(path) for tag in
           ('_deepdta', '_hyperattentiondti', '_moltrans', '_drugban')):
        continue                      # another model's cell, restored alongside ours
    r = json.load(open(path))
    rows.append((r['split'], r['seed'], r['test_metrics']['auroc'],
                 r['test_metrics']['auprc'], r.get('best_epoch', '-'),
                 r.get('resumed_after_epoch') or '-'))

print(f'{"level":12s} {"seed":>4s} {"AUROC":>7s} {"AUPRC":>7s} {"best":>5s} {"resumed":>8s}')
for level, seed, auroc, auprc, best, resumed in sorted(rows):
    print(f'{level:12s} {seed:>4} {auroc:>7.4f} {auprc:>7.4f} {str(best):>5s} {str(resumed):>8s}')
print(f'\n{len(rows)}/{TOTAL_CELLS} cells')

seen = {}
for level, seed, auroc, *_ in rows:
    key = (level, round(auroc, 6))
    if key in seen:
        print(f'!! {level} seed {seed} has the same AUROC as seed {seen[key]} -- is --seed '
              'reaching training? Do not merge these.')
    seen[key] = seed


## 10. Take the results with you

ColdSite-DTI's checkpoints are small (~2.5 MB), so this zip is megabytes, not gigabytes.

In [ ]:
import subprocess

RES_ZIP = f'{WORK}/coldsite_{DATASET}_results.zip'
finished = [p for p in glob.glob(f'{RESULTS}/*')
            if not os.path.basename(p).endswith('_resume.pt')]
resumes = glob.glob(f'{RESULTS}/*_resume.pt')

if os.path.exists(RES_ZIP):
    os.remove(RES_ZIP)
subprocess.run(['zip', '-q', '-j', RES_ZIP, *finished], check=True)
print(f'{os.path.basename(RES_ZIP)}: {len(finished)} file(s), '
      f'{os.path.getsize(RES_ZIP)/1e6:.0f} MB')

if resumes:
    RESUME_ZIP = f'{WORK}/coldsite_{DATASET}_resume.zip'
    if os.path.exists(RESUME_ZIP):
        os.remove(RESUME_ZIP)
    subprocess.run(['zip', '-q', '-j', RESUME_ZIP, *resumes], check=True)
    print(f'{os.path.basename(RESUME_ZIP)}: {len(resumes)} unfinished cell(s) -- upload '
          'this one too, or the next commit restarts them from epoch 1.')
